# 🛠️ EYE-D: Feasibility Check

# 1. yolov8n

In [ ]:
# 필요한 패키지 설치
!pip install ultralytics opencv-python matplotlib

### 1.1. 모델 로드

In [ ]:
from ultralytics import YOLO
import cv2
import matplotlib.pyplot as plt

# 사전 학습된 YOLOv8 nano 모델 로드 (가중치가 없으면 자동으로 다운로드 됨)
model = YOLO('yolov8n.pt')
print("YOLOv8 모델 로드 완료!")

### 1.2. 이미지 추론 (Inference)

In [ ]:
# 샘플 이미지 URL을 사용하여 추론 실행
image_url = 'https://ultralytics.com/images/bus.jpg'

# conf: 신뢰도 임계값 (0.25 이상의 정확도를 가진 객체만 탐지)
results = model.predict(source=image_url, conf=0.25)

# 예측 결과 시각화 준비
result = results[0]
img_with_boxes = result.plot() # 바운딩 박스가 그려진 넘파이 배열(BGR 형식) 반환

# Matplotlib를 이용하여 결과 출력 (BGR -> RGB 변환 필요)
plt.figure(figsize=(10, 8))
plt.imshow(cv2.cvtColor(img_with_boxes, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.show()

### 1.3. 결과 데이터 분석
예측된 객체의 클래스, 신뢰도(Confidence), 바운딩 박스 좌표를 추출해봅니다.

In [ ]:
boxes = result.boxes  # 바운딩 박스 정보 객체

for box in boxes:
    # 1. 클래스 ID 및 이름
    class_id = int(box.cls[0])
    class_name = model.names[class_id]
    
    # 2. 신뢰도 (Confidence Score)
    conf = float(box.conf[0])
    
    # 3. 바운딩 박스 좌표 (x1, y1, x2, y2)
    x1, y1, x2, y2 = map(int, box.xyxy[0])
    
    print(f"탐지됨: {class_name} (신뢰도: {conf:.2f}) | 좌표: [{x1}, {y1}, {x2}, {y2}]")

# 2. Multi-Object Tracking (MOT) Tutorial

이 노트북은 `boxmot` 라이브러리와 `ultralytics` YOLOv8을 활용하여 영상 내 다수의 객체를 탐지하고 각각에게 고유 ID를 부여하여 연속적으로 추적(Tracking)하는 기본적인 방법을 다룹니다.

In [ ]:
# 필요한 패키지 설치
!pip install ultralytics boxmot opencv-python matplotlib

### 2.1. 모델 및 트래커 로드
객체 탐지를 위한 YOLO 모델과, 객체 간 ID를 유지해 주는 ByteTrack 트래커를 초기화합니다.

In [ ]:
import sys
import boxmot

print(sys.executable)
print(boxmot.__file__)
print(boxmot.__version__)

### 2.2. 샘플 영상에 추적(Tracking) 적용
비디오 클립을 읽어 각 프레임마다 탐지(Detection) 후 트래커(Tracker)를 업데이트하는 루프를 구성해봅니다.

In [ ]:

from IPython.display import display, Image, clear_output

import cv2
import numpy as np
import torch
from boxmot.trackers.tracker_zoo import create_tracker
from ultralytics import YOLO

# 1. 초기 설정
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
half = True if device != 'cpu' else False
tracker = create_tracker('botsort', reid_weights='osnet_x0_25_msmt17.pt', device=device, half=half)
model = YOLO('yolov8n.pt') 
cap = cv2.VideoCapture('../../data/g1.mp4') # 노트북 파일 위치에 따라 경로 주의!

frame_count = 0

# 2. 실시간 스트리밍 루프
while cap.isOpened():
    ret, frame = cap.read()
    if not ret or frame_count > 50:
        break

    # 객체 탐지 및 트래킹
    results = model.predict(frame, conf=0.4, verbose=False)
    if len(results[0].boxes) > 0:
        dets = results[0].boxes.data.cpu().numpy()
    else:
        dets = np.empty((0, 6))

    tracks = tracker.update(dets, frame)
    # 시각화 (바운딩 박스)
    for track in tracks:
        x1, y1, x2, y2, track_id = map(int, track[:5])
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(frame, f"ID: {track_id}", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    # OpenCV 이미지를 Jupyter에서 렌더링하기 위해 JPG 포맷으로 인코딩
    _, jpeg = cv2.imencode('.jpg', frame)
    
    # 기존에 렌더링된 이미지를 지우고 새 프레임으로 덮어쓰기
    clear_output(wait=True)
    display(Image(data=jpeg.tobytes()))

    frame_count += 1

cap.release()

# 3. RE-ID Tutorial

* `torchreid` 라이브러리를 사용하여 탐지된 객체로부터 특징 벡터(Feature Vector)를 추출하는 방법을 다룹니다.

In [ ]:
# RE-ID 실습에 필요한 의존성 패키지를 설치합니다.
# tensorboard: torchreid 로깅용 의존성
# gdown: 사전 학습된 모델 다운로드용
# torchreid: Re-ID 핵심 라이브러리
!pip install tensorboard gdown torchreid

# 설치 후 커널을 재시작(Restart Kernel)해야 할 수도 있습니다.

### 3.1. Re-ID 모델 로드

In [ ]:
try:
    from torchreid.utils import FeatureExtractor
except ImportError:
    # 일부 버전에서는 reid 서브패키지 아래에 위치할 수 있습니다.
    from torchreid.reid.utils import FeatureExtractor

# OSNet-light 모델 로드 (market1501 데이터셋으로 학습된 가중치 사용)
reid_model = FeatureExtractor(
    model_name='osnet_x0_25',
    device='cpu'  # 또는 'cuda'
)
print("Re-ID 모델 로드 완료!")

### 3.2. 특징 벡터 추출

* 탐지된 사람의 이미지를 모델 입력 크기에 맞춰 리사이징한 후 특징 벡터를 추출합니다.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# 샘플 이미지 로드
img = cv2.imread('bus.jpg')

# 이미지에서 임의의 사람 영역 크롭 (예: YOLO 탐지 결과)
# bus.jpg에서 첫 번째 사람 영역 대략적인 좌표 [y1:y2, x1:x2]
person_roi = img[400:600, 100:200]

if person_roi.size > 0:
    # 모델 입력 크기(256, 128)로 리사이징
    person_roi_resized = cv2.resize(person_roi, (80, 80))
    
    # 특징 벡터 추출
    features = reid_model(person_roi_resized)
    
    print(f"추출된 벡터 차원: {features.shape}")
    print(f"벡터 샘플 (앞 10개): {features[0][:10]}")
    
    # 시각화
    plt.imshow(cv2.cvtColor(person_roi_resized, cv2.COLOR_BGR2RGB))
    plt.title("Extracted Person ROI")
    plt.axis('off')
    plt.show()
else:
    print("ROI 영역이 유효하지 않습니다.")

In [ ]:

from IPython.display import display, Image, clear_output

import cv2
import numpy as np
import torch
from boxmot.trackers.tracker_zoo import create_tracker
from ultralytics import YOLO

# 1. 초기 설정
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
half = True if device != 'cpu' else False
tracker = create_tracker('botsort', reid_weights='osnet_x0_25_msmt17.pt', device=device, half=half)
model = YOLO('yolov8n.pt') 
cap = cv2.VideoCapture('../../data/g1.mp4') # 노트북 파일 위치에 따라 경로 주의!

frame_count = 0

# 2. 실시간 스트리밍 루프
while cap.isOpened():
    ret, frame = cap.read()
    if not ret or frame_count > 50:
        break

    # 객체 탐지 및 트래킹
    results = model.predict(frame, conf=0.4, verbose=False)
    if len(results[0].boxes) > 0:
        dets = results[0].boxes.data.cpu().numpy()
    else:
        dets = np.empty((0, 6))

    tracks = tracker.update(dets, frame)
    # 시각화 (바운딩 박스)
    for track in tracks:
        x1, y1, x2, y2, track_id = map(int, track[:5])
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(frame, f"ID: {track_id}", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    # OpenCV 이미지를 Jupyter에서 렌더링하기 위해 JPG 포맷으로 인코딩
    _, jpeg = cv2.imencode('.jpg', frame)
    
    # 기존에 렌더링된 이미지를 지우고 새 프레임으로 덮어쓰기
    clear_output(wait=True)
    display(Image(data=jpeg.tobytes()))

    frame_count += 1

cap.release()

### 3.3. 실시간 Re-ID 결과 시각화

* YOLOv8 탐지기, DeepOCSORT 추적기, 그리고 OSNet Re-ID 모델을 결합하여 실시간으로 객체를 추적하고 특징 벡터를 추출합니다.
* 추출된 Re-ID 특징 벡터의 일부를 바운딩 박스 위에 표시하여 실시간 계산 여부를 확인합니다.

In [ ]:
from IPython.display import display, Image, clear_output
import os
import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import json
from pathlib import Path
from boxmot.trackers.tracker_zoo import create_tracker
from ultralytics import YOLO
try:
    from torchreid.utils import FeatureExtractor
except ImportError:
    from torchreid.reid.utils import FeatureExtractor

# 1. 모델 및 트래커 초기화
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
half = True if device != 'cpu' else False

model = YOLO('yolov8n.pt')
reid_model = FeatureExtractor(model_name='osnet_x0_25', device='cpu')
tracker = create_tracker('botsort', reid_weights='osnet_x0_25_msmt17.pt', device=device, half=half)

# 2. 비디오 및 저장 데이터 초기화
video_path = '../../data/g1.mp4'
if not os.path.exists(video_path):
    video_path = 'output.mp4'

cap = cv2.VideoCapture(video_path)
print(f"비디오 로드: {video_path}")

# 임베딩 데이터를 저장할 리스트
embedding_records = []

try:
    frame_count = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret or frame_count > 10: # 10프레임 제한
            break
        
        frame_count += 1
        results = model(frame, verbose=False)
        det = results[0].boxes.xyxy.cpu().numpy()
        conf = results[0].boxes.conf.cpu().numpy()
        cls = results[0].boxes.cls.cpu().numpy()
        
        mask = cls == 0
        if len(det[mask]) > 0:
            dets_with_cls = np.hstack((det[mask], conf[mask, np.newaxis], cls[mask, np.newaxis]))
            tracks = tracker.update(dets_with_cls, frame)
            
            for track in tracks:
                x1, y1, x2, y2, track_id = map(int, track[:5])
                
                roi = frame[y1:y2, x1:x2]
                if roi.size > 0:
                    roi_resized = cv2.resize(roi, (256, 128))
                    features = reid_model(roi_resized)
                    
                    # 텐서를 리스트로 변환하여 저장
                    vector = features[0].cpu().numpy().tolist()
                    embedding_records.append({
                        "frame": frame_count,
                        "track_id": int(track_id),
                        "embedding": vector
                    })
                    
                    # 시각화용 텍스트
                    feat_sample = vector[:3]
                    label = f"ID:{track_id} [{feat_sample[0]:.2f}, ...]"
                    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                    cv2.putText(frame, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
        
        # 시각화
        plt.figure(figsize=(10, 6))
        plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        plt.title(f"Re-ID Data Collection - Frame {frame_count}")
        plt.axis('off')
        display(plt.gcf())
        clear_output(wait=True)
        plt.close()

except Exception as e:
    print(f"오류 발생: {e}")
finally:
    cap.release()
    
    # 3. 데이터 파일 저장 및 요약 출력
    output_file = 'embeddings_results.json'
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(embedding_records, f, indent=4)
    
    print("\n" + "="*50)
    print(f"데이터 수집 완료! (총 {len(embedding_records)}개 레코드)")
    print(f"파일 저장 경로: {os.path.abspath(output_file)}")
    
    # ID별 요약 정보 출력
    unique_ids = sorted(list(set(r['track_id'] for r in embedding_records)))
    print(f"감지된 추적 ID 목록: {unique_ids}")
    
    if unique_ids:
        print("\n[ID별 임베딩 샘플 (첫 5개 값)]")
        for tid in unique_ids:
            # 해당 ID의 첫 번째 벡터 가져오기
            sample_vec = next(r['embedding'] for r in embedding_records if r['track_id'] == tid)
            print(f"Track ID {tid}: {sample_vec[:5]} ... (길이: {len(sample_vec)})")
    print("="*50)

### 3.4. re-id 테스트

| 모델 카테고리 | 모델명 (ID) | 주요 특징 | 장점 | 단점 |
| :--- | :--- | :--- | :--- | :--- |
| **ReID 특화** | **OSNet** | Omni-Scale 특성 학습 (다양한 크기의 패턴 추출) | 현시점 가장 균형 잡힌 모델. 성능 대비 연산량이 매우 적음. | 복잡한 배경이 섞인 환경에서 간혹 오탐 발생 가능. |
| **부분 기반** | **PCB** | 이미지를 수평으로 6등분하여 부분별 특징 학습 | 신체 부위별 세밀한 특징 포착 능력 우수. 정확도가 높음. | 연산량이 상대적으로 많고, 사람의 자세 변화에 민감함. |
| **경량화** | **MobileNetV2** | Depthwise Separable Convolution 활용 | 모바일 및 엣지 디바이스에서 압도적인 추론 속도. | 복잡한 의상이나 비슷한 색상 환경에서 식별 성능이 낮음. |
| **표준 백본** | **ResNet50** | 잔차 연결(Residual Connection) 기반의 표준 모델 | 안정적인 학습 성능. 사전 학습(Pre-trained) 가중치가 풍부함. | ReID 전용 모델(OSNet 등)에 비해 효율성이 떨어짐. |
| **도메인 강화** | **ResNet-IBN** | Instance & Batch Normalization 혼합 적용 | 조명 변화, 카메라 각도 변화 등 도메인 차이에 매우 강함. | 일반 ResNet 대비 메모리 사용량이 약간 더 많음. |

In [ ]:
import torchreid

# 사용 가능한 모든 모델 이름 출력
print(torchreid.models.show_avai_models())

# 특정 모델 생성 예시 (클래스 수에 맞춰 생성)
model = torchreid.models.build_model(
    name='osnet_x1_0',
    num_classes=751, # Market1501 데이터셋 기준
    loss='softmax',
    pretrained=True
)

# 4. Local Vector DB (Qdrant)

추출된 임베딩 벡터를 효율적으로 관리하고 검색하기 위해 벡터 데이터베이스인 **Qdrant**를 사용합니다. 
파일(JSON) 저장은 기록용으로 좋지만, 수만 명의 인물 데이터에서 유사한 인물을 50ms 이내에 찾아내기 위해서는 벡터 인덱싱 기술이 필수적입니다.

### 4.1. Qdrant 클라이언트 설치


In [ ]:
!pip install qdrant-client

### 4.2. Qdrant 초기화 및 컬렉션 생성
튜토리얼의 편의를 위해 별도의 서버 설치 없이 메모리(`:memory:`) 모드로 실행합니다.

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
import json
import os

# 1. Qdrant 클라이언트 초기화 (메모리 모드)
client = QdrantClient(":memory:")
COLLECTION_NAME = "person_reid"

# 2. 컬렉션 생성 (OSNet 특징 벡터 크기는 512차원)
client.recreate_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=512, distance=Distance.COSINE),
)
print(f"컬렉션 '{COLLECTION_NAME}'이 생성되었습니다.")

# 3. JSON 데이터 로드
input_file = 'embeddings_results.json'
if os.path.exists(input_file):
    with open(input_file, 'r') as f:
        embedding_data = json.load(f)
    
    # 4. Qdrant에 포인트(데이터) 삽입
    points = []
    for i, record in enumerate(embedding_data):
        points.append(PointStruct(
            id=i, 
            vector=record['embedding'], 
            payload={"track_id": record['track_id'], "frame": record['frame']}
        ))
    
    # 배치 삽입
    client.upsert(collection_name=COLLECTION_NAME, points=points)
    print(f"총 {len(points)}개의 벡터를 Qdrant에 저장했습니다.")
else:
    print("JSON 파일이 없습니다. 3번 섹션을 먼저 실행해 주세요.")


### 4.3. 유사 인물 검색 (Vector Search)
수집된 데이터 중 하나를 골라 '쿼리'로 사용하여, DB 내에서 가장 유사한 인물을 찾아봅니다.


In [ ]:
if len(embedding_data) > 0:
    # 테스트용 쿼리: 첫 번째 데이터의 벡터를 검색 쿼리로 사용
    query_vector = embedding_data[0]['embedding']
    target_id = embedding_data[0]['track_id']
    
    print(f"검색 쿼리 대상: Track ID {target_id}")
    
    try:
        # 방법 1: 최신 버전에서 권장되는 query_points 사용
        search_result = client.query_points(
            collection_name=COLLECTION_NAME,
            query=query_vector,
            limit=3
        ).points
    except AttributeError:
        # 방법 2: 기존 search 메서드 사용
        search_result = client.search(
            collection_name=COLLECTION_NAME,
            query_vector=query_vector,
            limit=3
        )

    print("\n[검색 결과]")
    for hit in search_result:
        # hit 객체의 구조에 따라 데이터 추출 (PointStruct 또는 ScoredPoint)
        payload = hit.payload
        score = hit.score
        print(f"ID: {payload['track_id']} (Frame {payload['frame']}) | 유사도: {score:.4f}")
else:
    print("검색할 데이터가 없습니다.")


### 4.5. Milvus Lite

**Milvus Lite**는 Milvus 벡터 DB의 경량화 버전으로, 별도의 서버나 컨테이너 없이 파이썬 라이브러리(`pymilvus`)만으로 로컬 환경에서 벡터 검색 기능을 제공합니다. 

- **장점**: 설치가 매우 간편함, 엣지 장비(Jetson) 리소스 최소화, Milvus 클라우드/서버와 호환됨.
- **용도**: 엣지 단에서의 실시간 Re-ID 검색 및 프로토타이핑.

In [ ]:
!pip install pymilvus 
!pip install milvus-lite

In [ ]:
from pymilvus import MilvusClient
import numpy as np

# 1. Milvus Lite 초기화 (로컬 파일 기반 저장)
db_name = "milvus_demo.db"
client = MilvusClient(db_name)

COLLECTION_NAME = "person_reid_test"

# 2. 기존 컬렉션이 있다면 삭제 (테스트용)
if client.has_collection(COLLECTION_NAME):
    client.drop_collection(COLLECTION_NAME)

# 3. 컬렉션 생성 (512차원 특징 벡터 기준)
client.create_collection(
    collection_name=COLLECTION_NAME,
    dimension=512,  # OSNet 특징 벡터 크기
)
print(f"컬렉션 '{COLLECTION_NAME}' 생성 완료.")

# 4. 가상 데이터 생성 및 삽입
# 실제 환경에서는 추출된 Re-ID 벡터를 사용합니다.
data = [
    {"id": i, "vector": np.random.random(512).tolist(), "subject_id": f"person_{i}"}
    for i in range(10)
]

res = client.insert(
    collection_name=COLLECTION_NAME,
    data=data
)
print(f"{len(data)}개의 데이터가 삽입되었습니다.")

# 5. 유사도 검색 수행
# 임의의 쿼리 벡터 생성
query_vectors = [np.random.random(512).tolist()]

res = client.search(
    collection_name=COLLECTION_NAME,
    data=query_vectors,
    limit=3,
    output_fields=["subject_id"] # 결과에 포함할 메타데이터 필드
)

print("\n[Milvus Lite 검색 결과]")
for result in res[0]:
    print(f"ID: {result['entity']['subject_id']} | 거리(Score): {result['distance']:.4f}")


* 리소스 비교: Qdrant와 Milvus Lite 중 Orin Nano의 메모리와 CPU를 더 적게 점유하는 라이브러리를 선택
* 영속성: Milvus Lite는 milvus_demo.db와 같은 단일 파일에 데이터를 저장하므로 관리가 편리합니다.

# 5. Production Core Modules Verification
이 단계부터는 실제 프로덕션 패키지인 `edge/src`에 구현된 코어 모듈들(`ImagePreprocessor`, `ReIDExtractor`, `PipelineRunner`)이 주피터 환경에서 유기적이고 안정적으로 임포트 및 상호 통신이 가능하지 실전 검증합니다.

### 5.1. 프로젝트 경로 설정 및 코어 모듈 임포트 검증
노트북 실행 위치인 `edge/notebooks/` 상위의 프로젝트 루트 `edge/`를 검색 경로로 등록하고, 구현한 핵심 비전 파이프라인 모듈들을 임포트합니다.

In [ ]:
import os
import sys

# 1. edge/ 상위 디렉토리를 파이썬 검색 경로에 추가합니다.
current_dir = os.getcwd()
project_edge_root = os.path.abspath(os.path.join(current_dir, ".."))
if project_edge_root not in sys.path:
    sys.path.append(project_edge_root)
    print(f"[✔] 프로젝트 루트 경로 등록 완료: {project_edge_root}")

# 2. 구현한 핵심 모듈들을 임포트합니다.
try:
    from src.core.preprocessor import ImagePreprocessor
    from src.core.reid_extractor import ReIDExtractor
    from src.core.pipeline_runner import PipelineRunner
    
    print("[✔] src 하위 모든 핵심 EYE-D 엣지 모듈 임포트 성공!")
except ImportError as e:
    print(f"[❌] 임포트 실패! 경로 설정을 다시 확인해주세요. 에러내용: {e}")

### 5.2. 이미지 전처리 엔진(ImagePreprocessor) 악조건 보정 테스트
감마 1.6 보정(야간 어두움 해소), CLAHE 4.0(역광 대비 조정), 언샤프 마스크(선명도 향상) 기능이 노트북 상에서 어떻게 이미지 밝기와 선명도를 복원하는지 Side-by-Side 플롯팅으로 검증합니다.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from src.core.preprocessor import ImagePreprocessor

# 1. 테스트용 어두운 암흑 프레임과 역광 모사 프레임 생성
np.random.seed(42)
test_frame = np.ones((300, 300, 3), dtype=np.uint8) * 15 # 매우 어두운 상태 (밝기 15)
# 중앙 부분에 보행자 모사 텍스처 삽입
test_frame[100:200, 100:200] = (np.random.rand(100, 100, 3) * 35).astype(np.uint8)

# 2. ImagePreprocessor 인스턴스 생성
preprocessor = ImagePreprocessor()

# 3. 야간(is_night=True) 상태로 전처리 보정 적용
enhanced_frame = preprocessor.process(test_frame, is_night=True, is_backlit=False)

# 4. ROI 영역 선명화(enhance_roi) 필터 추가 적용
roi_slice = test_frame[100:200, 100:200]
enhanced_roi = preprocessor.enhance_roi(roi_slice)

# 5. matplotlib을 이용해 보정 전/후 비교 플롯 출력
plt.figure(figsize=(12, 5))

plt.subplot(1, 3, 1)
plt.imshow(cv2.cvtColor(test_frame, cv2.COLOR_BGR2RGB))
plt.title("Original (Dark Night)")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(cv2.cvtColor(enhanced_frame, cv2.COLOR_BGR2RGB))
plt.title("Enhanced (Gamma 1.6 + CLAHE)")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(cv2.cvtColor(enhanced_roi, cv2.COLOR_BGR2RGB))
plt.title("Sharp ROI (Unsharp Mask)")
plt.axis("off")

plt.tight_layout()
plt.show()

print(f"보정 전 평균 밝기: {np.mean(test_frame):.2f}")
print(f"야간 보정 후 평균 밝기: {np.mean(enhanced_frame):.2f}")

### 5.3. OSNet Re-ID 임베딩 특징 추출 테스트
OSNet 추출기가 정상적으로 512차원의 L2 정규화 실수 특징 벡터를 PyTorch Fallback 환경에서 안정적으로 뽑아내는지 실성능을 검증합니다.

In [ ]:
import torch
from src.core.reid_extractor import ReIDExtractor
from src.core.tracker import TrackResult

# 1. Re-ID 특징 추출기 로딩 (onnx 가속 미사용 시 PyTorch Fallback 작동)
reid = ReIDExtractor(use_onnx=False)
print(f"ReIDExtractor 백엔드 로드 성공! (가중치 차원: {reid.vector_dim})")

# 2. 임의의 128x64 크기의 보행자 이미지 조각(ROI) 가상 생성
dummy_roi = (np.random.rand(128, 64, 3) * 255).astype(np.uint8)

# 3. Tracker에서 얻은 가상 트랙 결과 리스트 래핑
dummy_tracks = [
    TrackResult(track_id=42, bbox=[10, 10, 74, 138], confidence=0.89)
]

# 4. Re-ID 특징 벡터 추출
dummy_frame = np.zeros((300, 300, 3), dtype=np.uint8)
dummy_frame[10:138, 10:74] = dummy_roi

results = reid.extract(dummy_frame, dummy_tracks)

# 5. 추출된 특징 벡터 L2 정규성 검증
if len(results) > 0:
    record = results[0]
    vector = record['vector']
    print("\n[✔] Re-ID 특징 추출 성공!")
    print(f" - 대상 Track ID: {record['track_id']}")
    print(f" - 추출된 임베딩 차원수: {len(vector)} 차원")
    print(f" - 임베딩 벡터 샘플 (앞 5개): {vector[:5]}")
    
    l2_norm = np.linalg.norm(vector)
    print(f" - L2 Normalization 검증 (정규화 크기): {l2_norm:.4f} (1.0000에 수렴해야 함)")
else:
    print("[❌] 특징 추출에 실패했습니다.")

### 5.4. 전체 E2E 파이프라인 제어기(PipelineRunner) 가상 실행 테스트
객체 탐지, 추적, 보정, 특징 추출, 그리고 최종 Vector DB 통신 규격 적재까지 원스톱으로 흘러가는 통합 Runner 라이프사이클의 무결성을 진단합니다.

In [ ]:
from unittest.mock import MagicMock
from src.core.pipeline_runner import PipelineRunner

# 테스트를 위해 Mock 모듈 임포트 (경로 검증 및 Fallback 자동 보정)
try:
    from tests.harness.mocks import MockDetector, MockTracker, MockReIDExtractor
except ImportError:
    # 실행환경의 작업경로 유동성에 대비한 복구 로직
    sys.path.append(os.path.abspath(os.path.join(project_edge_root, "tests")))
    try:
        from harness.mocks import MockDetector, MockTracker, MockReIDExtractor
    except ImportError:
        sys.path.append(os.path.dirname(project_edge_root))
        from edge.tests.harness.mocks import MockDetector, MockTracker, MockReIDExtractor

# 1. Vector DB API 통신 모킹 셋업
mock_db = MagicMock()
sent_records = []
mock_db.upsert = MagicMock(side_effect=lambda col, records: sent_records.extend(records))

# 2. PipelineRunner 기동
runner = PipelineRunner(config={
    'collection_name': 'notebook_sandbox_db',
    'use_onnx': False
}, db_client=mock_db)

# 3. 표준 인터페이스 Mock 컴포넌트 강제 주입
runner._detector = MockDetector(num_detections=1)
runner._tracker = MockTracker(track_ids=[7])
runner._reid = MockReIDExtractor(vector_dim=512)
runner.running = True

# 4. 가상 프레임 흘려보내기
input_frame = np.ones((360, 640, 3), dtype=np.uint8) * 100
runner_result = runner.process_frame(input_frame, camera_id="cam_notebook_test")

# 5. 파이프라인의 종단 데이터 결과 출력
print("[✔] 파이프라인 1프레임 연쇄 실행 성공!")
print(f" - 카메라 ID: {runner_result['camera_id']}")
print(f" - 탐지된 인물 수: {len(runner_result['detections'])}")
print(f" - 추적 중인 트랙 수: {len(runner_result['tracks'])}")
print(f" - 생성된 임베딩 패킷 수: {len(runner_result['reid_vectors'])}")

if len(sent_records) > 0:
    print(f"\n[✔] DB 전송 버퍼에 최종 적재된 패킷 샘플:")
    print(f" - DB 저장 ID (UUID): {sent_records[0]['id']}")
    print(f" - 페이로드 정보: {sent_records[0]['payload']}")

# 6. E2E Real Video Visual Tracking Live Demonstration
실제 비디오 파일(`g1.mp4`)을 사용하여 모킹이 아닌 **실제 YOLOv8n 검출기, BoxMOT 트래커, OSNet Re-ID 추출기**의 실전 흐름을 검증하고, 프레임 내 사람들의 움직임 위에 고유 Track ID 라벨과 바운딩 박스를 형광 초록색으로 실시간 합성(OpenCV Annotation Draw)하여 주피터 브라우저 상에서 깜빡임이 없이 부드럽게(Flicker-Free Live Streaming) 출력합니다.

In [ ]:
# 1. 테스트 비디오 파일(g1.mp4) 경로 자동 탐색
candidate_paths = [
    "../../data/g1.mp4",
    "../data/g1.mp4",
    "./data/g1.mp4",
    os.path.abspath(os.path.join(project_edge_root, "data", "g1.mp4")),
]

video_path = None
for p in candidate_paths:
    if os.path.exists(p):
        video_path = os.path.abspath(p)
        break

if video_path is None:
    print("[❌] 에러: g1.mp4 비디오 파일을 찾을 수 없습니다.")
    print(" - 'edge/data/g1.mp4' 위치 혹은 data/ 아래에 테스트 비디오가 존재해야 합니다.")
else:
    print(f"[✔] 테스트 비디오 파일 발견: {video_path}")

In [ ]:
if video_path:
    # 2. 실제 PipelineRunner 가동 (use_onnx=False 로 PyTorch Fallback으로 가동)
    # db_client는 NullDBClient(기본값)를 사용하여 외부 DB 호출 없이 순수 분석 레이턴시를 측정합니다.
    print("\n[⏳] YOLO 모델 및 OSNet 가중치를 로드하고 파이프라인을 초기화합니다. 잠시만 기다려주세요...")
    
    runner = PipelineRunner(config={
        'yolo_model': 'yolov8n.pt',
        'conf_threshold': 0.4,
        'tracker_type': 'botsort',
        'reid_weights': 'osnet_x0_25_msmt17.pt',
        'reid_model': 'osnet_x0_25',
        'use_onnx': False
    })
    
    runner.start()
    print("[✔] 실제 파이프라인 오케스트레이터 가동 시작!")

In [ ]:
if video_path and runner.running:
    # 3. IPython display 모듈 로드
    from IPython.display import display, Image, clear_output
    import time
    
    # 비디오 캡처 객체 생성
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    print(f" - 비디오 속성: 총 {total_frames} 프레임, {fps:.2f} FPS")
    
    max_frames_to_test = 100  # 실시간 시각화 구동성 검증을 위해 100프레임으로 테스트 상향 조정!
    processed_count = 0
    detected_persons_total = 0
    embeddings_generated_total = 0
    
    print("\n" + "="*80)
    print(" ▶ E2E Pipeline Real Video Visual Tracking (YOLO ➔ BoxMOT ➔ OSNet ReID)")
    print(" ▶ 아래 프레임 영역에 실시간 트래킹 박스가 깜빡임 없이 부드럽게 렌더링됩니다.")
    print("="*80)
    
    # Flicker-free 출력을 위한 고유 디스플레이 핸들 홀더 선언
    display_handle = display(None, display_id=True)
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret or processed_count >= max_frames_to_test:
            break
            
        processed_count += 1
        
        # 파이프라인 관통 (전처리 ➔ YOLO 탐지 ➔ BoxMOT Tracker 추적 ➔ Re-ID 512차원 임베딩 생성)
        result = runner.process_frame(frame, camera_id="cam_g1_verification")
        
        # 현재 프레임 분석 결과 수집
        num_dets = len(result['detections'])
        num_tracks = len(result['tracks'])
        num_vectors = len(result['reid_vectors'])
        latency = result['processing_time_ms']
        
        detected_persons_total += num_dets
        embeddings_generated_total += num_vectors
        
        # 실시간 아노테이션(바운딩 박스 & 트랙 ID 드로잉) 적용
        annotated_frame = frame.copy()
        for track in result['tracks']:
            track_id = track['track_id']
            bbox = track['bbox']
            x1, y1, x2, y2 = map(int, bbox)
            
            # 1. 형광색 바운딩 박스 그리기 (BGR: 초록색)
            cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            
            # 2. 트랙 ID 텍스트 배경 라벨 그리기
            label = f"ID: {track_id}"
            (w, h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 2)
            cv2.rectangle(annotated_frame, (x1, y1 - 22), (x1 + w + 10, y1), (0, 255, 0), -1)
            
            # 3. 텍스트 라벨 쓰기
            cv2.putText(annotated_frame, label, (x1 + 5, y1 - 7),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 2, cv2.LINE_AA)
        
        # 프레임 좌측 상단에 실시간 성능 리포트(프레임 인덱스, 객체 수, 지연시간) 오버레이
        info_text = f"Frame: #{processed_count:02d} | Dets: {num_dets} | Tracks: {num_tracks} | Latency: {latency:.1f}ms"
        cv2.putText(annotated_frame, info_text, (15, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2, cv2.LINE_AA)
        
        # BGR ➔ JPEG 이미지 압축 및 디스플레이 갱신 (부드러운 실시간 렌더링 작동)
        _, jpeg = cv2.imencode('.jpg', annotated_frame)
        display_handle.update(Image(data=jpeg.tobytes()))
        
        # 주피터 디스플레이 동적 업데이트 조절을 위한 초미세 틱 타임 확보
        time.sleep(0.01)
              
        # 최초 1개 특징 벡터가 성공적으로 생성되었을 때 상세 패킷 데이터 구조 검증 리포트
        if num_vectors > 0 and processed_count == 1:
            sample_vector = result['reid_vectors'][0]
            print(f"\n[프레임 #1 패킷 검증] Track ID: {sample_vector['track_id']} | L2 Norm: {np.linalg.norm(sample_vector['vector']):.4f}")
            
    cap.release()
    runner.stop()
    
    print("\n" + "="*80)
    print(" 🎉 실제 비디오 E2E 파이프라인 실시간 시각화 검증 완료!")
    print(f"  - 총 테스트한 프레임: {processed_count} 프레임")
    print(f"  - 연쇄 탐지된 인물 수 합계: {detected_persons_total}명")
    print(f"  - 최종 생성 및 L2 정규화된 임베딩 벡터 수: {embeddings_generated_total}개")
    print("="*80)